# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 軌道データから、Tyganenko04のfield lineをplot

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

earth_radius = 6378.1  # km

psp.projects.themis.state(trange=trange, probe='a')
psp.cotrans(name_in='tha_pos_gse', name_out='tha_pos_sm', coord_in='gse', coord_out='sm')   # 'tha_pos_sm'

themis_a_pos_sm = psp.get_data('tha_pos_sm', xarray=True)
themis_a_pos_sm.values = themis_a_pos_sm.values / earth_radius  # convert to RE
themis_a_pos_sm.attrs['Units'] = 'R_E'
# trangeに合わせてデータを切り出し
themis_a_pos_sm = themis_a_pos_sm.sel(time=slice(trange[0], trange[1]))

print(themis_a_pos_sm)

ergpy.orb(trange=trange, level='l2', datatype='def')  # 'erg_orb_l2_pos_sm'
Arase_pos_sm = psp.get_data('erg_orb_l2_pos_sm', xarray=True)
Arase_pos_sm = Arase_pos_sm.sel(time=slice(trange[0], trange[1]))

print(Arase_pos_sm)

In [ ]:
Arase_pos_sm_x, Arase_pos_sm_y, Arase_pos_sm_z = Arase_pos_sm.values[:,0], Arase_pos_sm.values[:,1], Arase_pos_sm.values[:,2]
Arase_pos_sm_time = Arase_pos_sm.time
themis_a_pos_sm_x, themis_a_pos_sm_y, themis_a_pos_sm_z = themis_a_pos_sm.values[:,0], themis_a_pos_sm.values[:,1], themis_a_pos_sm.values[:,2]
themis_a_pos_sm_time = themis_a_pos_sm.time

In [ ]:
Arase_rmlatmlt_R = np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0 + Arase_pos_sm_z**2E0)
Arase_rmlatmlt_MLAT = np.rad2deg(np.arctan2(Arase_pos_sm_z, np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0)))
Arase_rmlatmlt_MLT = np.rad2deg(np.arctan2(Arase_pos_sm_y, Arase_pos_sm_x)) / 15. + 12.

Arase_rmlatmlt_L    = Arase_rmlatmlt_R / np.cos(np.deg2rad(Arase_rmlatmlt_MLAT))**2E0
Arase_rmlatmlt_L_x  = Arase_rmlatmlt_L / np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0) * Arase_pos_sm_x
Arase_rmlatmlt_L_y  = Arase_rmlatmlt_L / np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0) * Arase_pos_sm_y

THA_rmlatmlt_R = np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0 + themis_a_pos_sm_z**2E0)
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(themis_a_pos_sm_z, np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(themis_a_pos_sm_y, themis_a_pos_sm_x)) / 15. + 12.

THA_rmlatmlt_L    = THA_rmlatmlt_R / np.cos(np.deg2rad(THA_rmlatmlt_MLAT))**2E0
THA_rmlatmlt_L_x  = THA_rmlatmlt_L / np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0) * themis_a_pos_sm_x
THA_rmlatmlt_L_y  = THA_rmlatmlt_L / np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0) * themis_a_pos_sm_y

In [ ]:
# 文字の大きさ
mpl.rcParams['font.size'] = 25

In [ ]:
def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

fig = plt.figure(figsize=(11, 11), dpi=300)

ax1 = fig.add_subplot(221)
ax1.plot(themis_a_pos_sm_x, themis_a_pos_sm_y, color='magenta', label='THEMIS-A', linewidth=3)
ax1.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax1.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax1.plot(Arase_pos_sm_x, Arase_pos_sm_y, color='green', label='Arase', linewidth=3)
ax1.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax1.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -9)
ax1.set_ylim(8.5, -1.5)
ax1.set_title('(X, Y)')


ax2 = fig.add_subplot(222)
ax2.plot(themis_a_pos_sm_x, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax2.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax2.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax2.plot(Arase_pos_sm_x, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax2.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax2.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax2, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Z‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax2.set_xlim(1, -9)
ax2.set_ylim(-3, 7)
ax2.set_title('(X, Z)')

ax3 = fig.add_subplot(223)
ax3.plot(themis_a_pos_sm_y, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax3.scatter(themis_a_pos_sm_y[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax3.scatter(themis_a_pos_sm_y[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax3.plot(Arase_pos_sm_y, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax3.scatter(Arase_pos_sm_y[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax3.scatter(Arase_pos_sm_y[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax3, 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Z‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax3.set_xlim(5, -1)
ax3.set_ylim(-1, 5)
ax3.set_title('(Y, Z)')

ax4 = fig.add_subplot(224)
ax4.plot(THA_rmlatmlt_L_x, THA_rmlatmlt_L_y, color='magenta', label='THEMIS-A', linewidth=3)
ax4.scatter(THA_rmlatmlt_L_x[0],  THA_rmlatmlt_L_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax4.scatter(THA_rmlatmlt_L_x[-1], THA_rmlatmlt_L_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax4.plot(Arase_rmlatmlt_L_x, Arase_rmlatmlt_L_y, color='green', label='Arase', linewidth=3)
ax4.scatter(Arase_rmlatmlt_L_x[0],  Arase_rmlatmlt_L_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax4.scatter(Arase_rmlatmlt_L_x[-1], Arase_rmlatmlt_L_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax4, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax4.set_xlim(1, -9)
ax4.set_ylim(8.5, -1.5)
ax4.set_title(r'(X, Y) at MLAT = $0^{\circ}$')

for ax in (ax1, ax2, ax3, ax4):
    th = np.linspace(0, 2*np.pi, 256)
    ax.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
    if ax == ax1 or ax == ax2 or ax == ax4:
        # 円の中のx<0の部分を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
    if ax == ax3:
        # 円の中を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)>0), color='k')

    if ax == ax1:
        ax.text(-0.25, 1, '(a)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax2:
        ax.text(-0.25, 1, '(b)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax3:
        ax.text(-0.25, 1, '(c)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax4:
        ax.text(-0.25, 1, '(d)', transform=ax.transAxes, verticalalignment='top')

fig.tight_layout()
plt.show()

#fig.savefig(r"/mnt/j/KAW_observation/Figure_1_a.pdf")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt

import geopack
from geopack import trace_vectorized, smgsm_vectorized
from pyspedas.geopack.get_tsy_params import get_tsy_params

earth_radius = 6378.1  # km
trange = ['2022-09-01/22:25', '2022-09-01/23:15']


# ============================================================
# 1. GSM positions を取得する
#    trace_vectorized には SM ではなく GSM を渡す
# ============================================================

# THEMIS-A: tha_pos_gsm は通常 km
psp.projects.themis.state(trange=trange, probe='a')
tha_gsm = psp.get_data('tha_pos_gsm', xarray=True)
tha_gsm = tha_gsm.sel(time=slice(trange[0], trange[1]))
tha_gsm_re = tha_gsm.copy()
tha_gsm_re.values = tha_gsm.values / earth_radius
tha_gsm_re.attrs['Units'] = 'R_E'

# Arase: erg_orb_l2_pos_gsm が使えるならそれを使う
ergpy.orb(trange=trange, level='l2', datatype='def')
ara_gsm = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)
ara_gsm = ara_gsm.sel(time=slice(trange[0], trange[1]))

# Arase の単位が km か R_E か環境によって確認した方がよい
# 値の大きさで km っぽければ R_E に直す
ara_gsm_re = ara_gsm.copy()
r_med = np.nanmedian(np.linalg.norm(ara_gsm.values, axis=1))
if r_med > 100.0:
    ara_gsm_re.values = ara_gsm.values / earth_radius
else:
    ara_gsm_re.values = ara_gsm.values
ara_gsm_re.attrs['Units'] = 'R_E'

In [ ]:
# ============================================================
# 2. THEMIS-A と Arase を共通時刻へ補間し、中間位置を作る
# ============================================================

trace_times = pd.DatetimeIndex([pd.Timestamp('2022-09-01 22:50:00')])

tha_gsm_mid = tha_gsm_re.interp(time=trace_times)
tha_gsm_mid.name = 'tha_pos_gsm_mid'

arase_gsm_mid   = ara_gsm_re.interp(time=trace_times)
arase_gsm_mid.name = 'arase_pos_gsm_mid'

In [ ]:
# ============================================================
# 3. T04 / TS04 の parmod を作る
# ============================================================

# 少し広めにロードしておく
trange_param = ['2022-09-01/00:00', '2022-09-02/00:00']

psp.projects.kyoto.dst(trange=trange_param)
psp.projects.omni.data(trange=trange_param)

psp.join_vec(['BX_GSE', 'BY_GSM', 'BZ_GSM'])

params_name = get_tsy_params(
    dst_tvar='kyoto_dst',
    imf_tvar='BX_GSE-BY_GSM-BZ_GSM_joined',
    Np_tvar='proton_density',
    Vp_tvar='flow_speed',
    model='ts04',
    pressure_tvar='Pressure',
    speed=True,
    newname='ts04_par'
)

ts04_par = psp.get_data(params_name, xarray=True)
print(ts04_par)

# trace 時刻へ補間
ts04_par_i = ts04_par.interp(time=trace_times, method='nearest')

print(ts04_par_i)
print("parmod columns = [Pdyn, Dst, ByIMF, BzIMF, W1, W2, W3, W4, W5, W6]")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import geopack
from geopack import trace_vectorized, smgsm_vectorized


def datetime64_to_unix_seconds(t):
    """
    pandas.Timestamp, numpy.datetime64, str などを Unix seconds に変換する。
    geopack.recalc に渡す用。
    """
    return pd.Timestamp(t).timestamp()


def unpack_path(xx, yy, zz):
    """
    trace_vectorized(return_full_path=True) の戻り値を 1 本分の path に整形する。
    masked array でも ndarray でも処理できるようにする。
    """
    xx = np.ma.asarray(xx).squeeze()
    yy = np.ma.asarray(yy).squeeze()
    zz = np.ma.asarray(zz).squeeze()

    mask = (
        np.ma.getmaskarray(xx)
        | np.ma.getmaskarray(yy)
        | np.ma.getmaskarray(zz)
        | ~np.isfinite(np.ma.filled(xx, np.nan))
        | ~np.isfinite(np.ma.filled(yy, np.nan))
        | ~np.isfinite(np.ma.filled(zz, np.nan))
    )

    x = np.ma.filled(xx, np.nan)[~mask]
    y = np.ma.filled(yy, np.nan)[~mask]
    z = np.ma.filled(zz, np.nan)[~mask]

    return x, y, z


def _get_time_values(da):
    """
    xarray.DataArray から time 座標を取り出す。
    """
    if "time" not in da.coords:
        raise ValueError("Input position DataArray must have a 'time' coordinate.")
    return pd.DatetimeIndex(pd.to_datetime(da["time"].values))


def _interp_position_to_trace_times(pos_gsm, trace_times):
    """
    pos_gsm を trace_times に補間する。
    pos_gsm: xarray.DataArray(time, 3), GSM, R_E
    """
    trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))

    if not isinstance(pos_gsm, xr.DataArray):
        raise TypeError("pos_gsm must be an xarray.DataArray.")

    if pos_gsm.ndim != 2 or pos_gsm.shape[1] != 3:
        raise ValueError("pos_gsm must have shape (time, 3).")

    pos_i = pos_gsm.interp(time=trace_times, method="nearest")

    return pos_i


def _interp_parmod_to_trace_times(ts04_par, trace_times):
    """
    ts04_par を trace_times に補間する。
    ts04_par: xarray.DataArray(time, 10) or ndarray(len(trace_times), 10)
    """
    trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))

    if isinstance(ts04_par, xr.DataArray):
        if "time" in ts04_par.coords:
            par_i = ts04_par.interp(time=trace_times, method="nearest")
            par_values = np.asarray(par_i.values, dtype=float)
        else:
            par_values = np.asarray(ts04_par.values, dtype=float)
    else:
        par_values = np.asarray(ts04_par, dtype=float)

    if par_values.ndim == 1:
        if par_values.size != 10:
            raise ValueError("1D parmod must have length 10.")
        par_values = np.tile(par_values, (len(trace_times), 1))

    if par_values.shape != (len(trace_times), 10):
        raise ValueError(
            f"parmod shape must be ({len(trace_times)}, 10), "
            f"but got {par_values.shape}."
        )

    return par_values


def trace_field_lines_gsm_to_sm(
    pos_gsm,
    ts04_par,
    trace_times=None,
    sat_name=None,
    exname="t04",
    inname="igrf",
    rlim=30.0,
    r0=1.015,
    maxloop=5000,
    directions=(+1, -1),
    strict_scalar_models=False,
    verbose=True,
):
    """
    GSM 座標の衛星位置から field line を trace し、GSM path と SM path を返す。

    Parameters
    ----------
    pos_gsm : xarray.DataArray
        衛星位置。shape は (time, 3)。
        座標系は GSM、単位は R_E を想定。
    ts04_par : xarray.DataArray or ndarray
        T04/TS04 parmod。
        shape は (time, 10) または (10,)。
        成分順は [Pdyn, Dst, ByIMF, BzIMF, W1, W2, W3, W4, W5, W6]。
    trace_times : array-like or None
        trace する時刻。
        None の場合は pos_gsm.time をそのまま使う。
    sat_name : str or None
        出力 dict に入れる衛星名。
    exname : str
        外部磁場モデル。T04 の場合は "t04"。
    inname : str
        内部磁場モデル。"igrf" または "dip"。
    rlim : float
        外側境界 [R_E]。
    r0 : float
        内側境界 [R_E]。
    maxloop : int
        trace_vectorized の最大 step 数。
    directions : tuple
        trace 方向。通常は (+1, -1)。
    strict_scalar_models : bool
        geopack-vectorize の strict_scalar_models に渡す。
    verbose : bool
        True の場合、各時刻の status を print する。

    Returns
    -------
    field_lines : list[dict]
        各時刻の field line。
        各 dict は time, x_gsm, y_gsm, z_gsm, x_sm, y_sm, z_sm,
        x0_gsm, y0_gsm, z0_gsm, x0_sm, y0_sm, z0_sm, status, parmod を持つ。
    """

    if trace_times is None:
        trace_times = _get_time_values(pos_gsm)
    else:
        trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))

    pos_i = _interp_position_to_trace_times(pos_gsm, trace_times)
    par_values = _interp_parmod_to_trace_times(ts04_par, trace_times)

    field_lines = []

    for it, t in enumerate(trace_times):
        x0, y0, z0 = np.asarray(pos_i.values[it, :], dtype=float)
        parmod = np.asarray(par_values[it, :], dtype=float)

        if not np.all(np.isfinite([x0, y0, z0])) or not np.all(np.isfinite(parmod)):
            if verbose:
                print(f"skip {t}: NaN in position or parmod")
                print("position =", [x0, y0, z0])
                print("parmod   =", parmod)
            continue

        geopack.recalc(datetime64_to_unix_seconds(t))

        paths_gsm = []

        for direction in directions:
            try:
                ret = trace_vectorized(
                    x0, y0, z0,
                    dir=direction,
                    rlim=rlim,
                    r0=r0,
                    parmod=parmod,
                    exname=exname,
                    inname=inname,
                    maxloop=maxloop,
                    return_full_path=True,
                    strict_scalar_models=strict_scalar_models
                )

                # geopack-vectorize の版差対策:
                # こちらの環境では
                # xf, yf, zf, xx, yy, zz, status
                # ただし別実装では status の位置が違う可能性がある。
                if len(ret) >= 7:
                    xf, yf, zf, xx, yy, zz, status = ret[:7]
                else:
                    raise RuntimeError(
                        f"Unexpected trace_vectorized return length: {len(ret)}"
                    )

                x_path, y_path, z_path = unpack_path(xx, yy, zz)

                if len(x_path) > 1:
                    paths_gsm.append(
                        (
                            x_path,
                            y_path,
                            z_path,
                            int(np.asarray(status).squeeze())
                        )
                    )

            except Exception as e:
                if verbose:
                    print(f"trace failed {t}, dir={direction}: {e}")

        if len(paths_gsm) == 0:
            if verbose:
                print(f"failed {t}: no valid path")
            continue

        # 2方向を結合する。
        # directions=(+1, -1) を想定して、後半の path を反転してつなぐ。
        if len(paths_gsm) == 2:
            x1, y1, z1, s1 = paths_gsm[0]
            x2, y2, z2, s2 = paths_gsm[1]

            x_gsm = np.r_[x2[::-1], x1[1:]]
            y_gsm = np.r_[y2[::-1], y1[1:]]
            z_gsm = np.r_[z2[::-1], z1[1:]]
            status = (s1, s2)

        else:
            x_gsm, y_gsm, z_gsm, s1 = paths_gsm[0]
            status = (s1,)

        # GSM -> SM
        geopack.recalc(datetime64_to_unix_seconds(t))
        x_sm, y_sm, z_sm = smgsm_vectorized(x_gsm, y_gsm, z_gsm, j=-1)
        x0_sm, y0_sm, z0_sm = smgsm_vectorized(x0, y0, z0, j=-1)

        field_lines.append({
            "sat_name": sat_name,
            "time": t,
            "x_gsm": np.asarray(x_gsm),
            "y_gsm": np.asarray(y_gsm),
            "z_gsm": np.asarray(z_gsm),
            "x_sm": np.asarray(x_sm),
            "y_sm": np.asarray(y_sm),
            "z_sm": np.asarray(z_sm),
            "x0_gsm": float(x0),
            "y0_gsm": float(y0),
            "z0_gsm": float(z0),
            "x0_sm": float(np.asarray(x0_sm)),
            "y0_sm": float(np.asarray(y0_sm)),
            "z0_sm": float(np.asarray(z0_sm)),
            "status": status,
            "parmod": parmod
        })

        if verbose:
            prefix = f"{sat_name}: " if sat_name is not None else ""
            print(prefix + f"{t} status {status} npts {len(x_sm)}")

    return field_lines

In [ ]:
field_line_tha  = trace_field_lines_gsm_to_sm(
    pos_gsm=tha_gsm_mid,
    ts04_par=ts04_par_i,
    trace_times=trace_times,
    sat_name="THEMIS-A",
    exname="t04",
    inname="igrf",
    rlim=30.0,
    maxloop=5000,
    verbose=True
)

field_line_arase = trace_field_lines_gsm_to_sm(
    pos_gsm=arase_gsm_mid,
    ts04_par=ts04_par_i,
    trace_times=trace_times,
    sat_name="Arase",
    exname="t04",
    inname="igrf",
    rlim=30.0,
    maxloop=5000,
    verbose=True
)

In [ ]:
# ============================================================
# Field lines for each spacecraft
# ============================================================

def plot_field_lines_on_3panels(
    field_lines,
    ax_xy,
    ax_xz,
    ax_yz,
    color,
    label=None,
    lw=1.2,
    alpha=0.75,
    ls='--',
    zorder=2,
    mark_footpoints=False,
):
    """
    field_lines の x_sm, y_sm, z_sm を
    (X,Y), (X,Z), (Y,Z) の3 panel に重ねる。
    label は最初の1本だけに付ける。
    """
    for i, fl in enumerate(field_lines):
        this_label = label if i == 0 else None

        ax_xy.plot(
            fl["x_sm"], fl["y_sm"],
            color=color, linestyle=ls, lw=lw, alpha=alpha,
            label=this_label, zorder=zorder
        )
        ax_xz.plot(
            fl["x_sm"], fl["z_sm"],
            color=color, linestyle=ls, lw=lw, alpha=alpha,
            zorder=zorder
        )
        ax_yz.plot(
            fl["y_sm"], fl["z_sm"],
            color=color, linestyle=ls, lw=lw, alpha=alpha,
            zorder=zorder
        )

        if mark_footpoints:
            # 両端を小さく表示。不要なら False のままでよい。
            ax_xy.scatter(
                [fl["x_sm"][0], fl["x_sm"][-1]],
                [fl["y_sm"][0], fl["y_sm"][-1]],
                color=color, s=12, alpha=alpha, zorder=zorder + 1
            )
            ax_xz.scatter(
                [fl["x_sm"][0], fl["x_sm"][-1]],
                [fl["z_sm"][0], fl["z_sm"][-1]],
                color=color, s=12, alpha=alpha, zorder=zorder + 1
            )
            ax_yz.scatter(
                [fl["y_sm"][0], fl["y_sm"][-1]],
                [fl["z_sm"][0], fl["z_sm"][-1]],
                color=color, s=12, alpha=alpha, zorder=zorder + 1
            )

In [ ]:
def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

fig = plt.figure(figsize=(11, 11), dpi=300)

ax1 = fig.add_subplot(221)
ax1.plot(themis_a_pos_sm_x, themis_a_pos_sm_y, color='magenta', label='THEMIS-A', linewidth=3)
ax1.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax1.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax1.plot(Arase_pos_sm_x, Arase_pos_sm_y, color='green', label='Arase', linewidth=3)
ax1.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax1.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒SM '+ r'[$R_{\mathrm{E}}$]', 'Y‒SM '+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -11)
ax1.set_ylim(11, -1)
ax1.set_title('(X, Y)')


ax2 = fig.add_subplot(222)
ax2.plot(themis_a_pos_sm_x, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax2.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax2.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax2.plot(Arase_pos_sm_x, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax2.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax2.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax2, 'X‒SM '+ r'[$R_{\mathrm{E}}$]', 'Z‒SM '+ r'[$R_{\mathrm{E}}$]')
ax2.set_xlim(1, -11)
ax2.set_ylim(-6, 6)
ax2.set_title('(X, Z)')

ax3 = fig.add_subplot(223)
ax3.plot(themis_a_pos_sm_y, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax3.scatter(themis_a_pos_sm_y[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax3.scatter(themis_a_pos_sm_y[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax3.plot(Arase_pos_sm_y, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax3.scatter(Arase_pos_sm_y[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax3.scatter(Arase_pos_sm_y[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax3, 'Y‒SM '+ r'[$R_{\mathrm{E}}$]', 'Z‒SM '+ r'[$R_{\mathrm{E}}$]')
ax3.set_xlim(10, -1)
ax3.set_ylim(-5.5, 5.5)
ax3.set_title('(Y, Z)')

for ax in (ax1, ax2, ax3, ax4):
    th = np.linspace(0, 2*np.pi, 256)
    ax.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
    if ax == ax1 or ax == ax2 or ax == ax4:
        # 円の中のx<0の部分を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
    if ax == ax3:
        # 円の中を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)>0), color='k')

    if ax == ax1:
        ax.text(-0.25, 1, '(a)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax2:
        ax.text(-0.25, 1, '(b)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax3:
        ax.text(-0.25, 1, '(c)', transform=ax.transAxes, verticalalignment='top')

plot_field_lines_on_3panels(
    field_line_tha,
    ax1, ax2, ax3,
    color='magenta',
    label='THEMIS-A field line',
    lw=1.1,
    alpha=0.65,
    ls='--',
    zorder=2
)

plot_field_lines_on_3panels(
    field_line_arase,
    ax1, ax2, ax3,
    color='green',
    label='Arase field line',
    lw=1.1,
    alpha=0.65,
    ls='--',
    zorder=2
)

fig.tight_layout()
plt.show()

fig.savefig(r"/mnt/j/KAW_observation/probe_trajectory_T04_sm.pdf")
fig.savefig(r"/mnt/j/KAW_observation/probe_trajectory_T04_sm.png")

# GSM座標系でplotする

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

earth_radius = 6378.1  # km

psp.projects.themis.state(trange=trange, probe='a')
psp.cotrans(name_in='tha_pos_gse', name_out='tha_pos_gsm', coord_in='gse', coord_out='gsm')   # 'tha_pos_gsm'

themis_a_pos_gsm = psp.get_data('tha_pos_gsm', xarray=True)
themis_a_pos_gsm.values = themis_a_pos_gsm.values / earth_radius  # convert to RE
themis_a_pos_gsm.attrs['Units'] = 'R_E'
# trangeに合わせてデータを切り出し
themis_a_pos_gsm = themis_a_pos_gsm.sel(time=slice(trange[0], trange[1]))

print(themis_a_pos_gsm)

ergpy.orb(trange=trange, level='l2', datatype='def')  # 'erg_orb_l2_pos_gsm'
Arase_pos_gsm = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)
Arase_pos_gsm = Arase_pos_gsm.sel(time=slice(trange[0], trange[1]))

print(Arase_pos_gsm)

In [ ]:
Arase_pos_gsm_x, Arase_pos_gsm_y, Arase_pos_gsm_z = Arase_pos_gsm.values[:,0], Arase_pos_gsm.values[:,1], Arase_pos_gsm.values[:,2]
Arase_pos_gsm_time = Arase_pos_gsm.time
themis_a_pos_gsm_x, themis_a_pos_gsm_y, themis_a_pos_gsm_z = themis_a_pos_gsm.values[:,0], themis_a_pos_gsm.values[:,1], themis_a_pos_gsm.values[:,2]
themis_a_pos_gsm_time = themis_a_pos_gsm.time

In [ ]:
# 文字の大きさ
mpl.rcParams['font.size'] = 25

In [ ]:
def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

fig = plt.figure(figsize=(11, 11), dpi=300)

ax1 = fig.add_subplot(221)
ax1.plot(themis_a_pos_gsm_x, themis_a_pos_gsm_y, color='magenta', label='THEMIS-A', linewidth=3)
ax1.scatter(themis_a_pos_gsm_x[0],  themis_a_pos_gsm_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax1.scatter(themis_a_pos_gsm_x[-1], themis_a_pos_gsm_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax1.plot(Arase_pos_gsm_x, Arase_pos_gsm_y, color='green', label='Arase', linewidth=3)
ax1.scatter(Arase_pos_gsm_x[0],  Arase_pos_gsm_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax1.scatter(Arase_pos_gsm_x[-1], Arase_pos_gsm_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒GSM '+ r'[$R_{\mathrm{E}}$]', 'Y‒GSM '+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -9)
ax1.set_ylim(8.5, -1.5)
ax1.set_title('(X, Y)')


ax2 = fig.add_subplot(222)
ax2.plot(themis_a_pos_gsm_x, themis_a_pos_gsm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax2.scatter(themis_a_pos_gsm_x[0],  themis_a_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax2.scatter(themis_a_pos_gsm_x[-1], themis_a_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax2.plot(Arase_pos_gsm_x, Arase_pos_gsm_z, color='green', label='Arase', linewidth=3)
ax2.scatter(Arase_pos_gsm_x[0],  Arase_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax2.scatter(Arase_pos_gsm_x[-1], Arase_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax2, 'X‒GSM '+ r'[$R_{\mathrm{E}}$]', 'Z‒GSM '+ r'[$R_{\mathrm{E}}$]')
ax2.set_xlim(1, -9)
ax2.set_ylim(-3, 7)
ax2.set_title('(X, Z)')

ax3 = fig.add_subplot(223)
ax3.plot(themis_a_pos_gsm_y, themis_a_pos_gsm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax3.scatter(themis_a_pos_gsm_y[0],  themis_a_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax3.scatter(themis_a_pos_gsm_y[-1], themis_a_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax3.plot(Arase_pos_gsm_y, Arase_pos_gsm_z, color='green', label='Arase', linewidth=3)
ax3.scatter(Arase_pos_gsm_y[0],  Arase_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax3.scatter(Arase_pos_gsm_y[-1], Arase_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax3, 'Y‒GSM '+ r'[$R_{\mathrm{E}}$]', 'Z‒GSM '+ r'[$R_{\mathrm{E}}$]')
ax3.set_xlim(5, -1)
ax3.set_ylim(-1, 5)
ax3.set_title('(Y, Z)')

for ax in (ax1, ax2, ax3):
    th = np.linspace(0, 2*np.pi, 256)
    ax.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
    if ax == ax1 or ax == ax2 or ax == ax4:
        # 円の中のx<0の部分を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
    if ax == ax3:
        # 円の中を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)>0), color='k')

    if ax == ax1:
        ax.text(-0.25, 1, '(a)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax2:
        ax.text(-0.25, 1, '(b)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax3:
        ax.text(-0.25, 1, '(c)', transform=ax.transAxes, verticalalignment='top')

fig.tight_layout()
plt.show()

#fig.savefig(r"/mnt/j/KAW_observation/Figure_1_a.pdf")

In [ ]:
# ============================================================
# 2. THEMIS-A と Arase を共通時刻へ補間し、中間位置を作る
# ============================================================

trace_times = pd.DatetimeIndex([pd.Timestamp('2022-09-01 22:50:00')])

themis_a_pos_gsm_mid = themis_a_pos_gsm.interp(time=trace_times)
themis_a_pos_gsm_mid.name = 'tha_pos_gsm_mid'

Arase_pos_gsm_mid   = Arase_pos_gsm.interp(time=trace_times)
Arase_pos_gsm_mid.name = 'Arase_pos_gsm_mid'

In [ ]:
print(themis_a_pos_gsm_mid)
print(np.sqrt(np.sum((themis_a_pos_gsm_mid.data)**2.)))
print(Arase_pos_gsm_mid)
print(np.sqrt(np.sum((Arase_pos_gsm_mid.data)**2.)))

In [ ]:
# ============================================================
# 3. T04 / TS04 の parmod を作る
# ============================================================

# 少し広めにロードしておく
trange_param = ['2022-09-01/00:00', '2022-09-02/00:00']

psp.projects.kyoto.dst(trange=trange_param)
psp.projects.omni.data(trange=trange_param)

psp.join_vec(['BX_GSE', 'BY_GSM', 'BZ_GSM'])

params_name = get_tsy_params(
    dst_tvar='kyoto_dst',
    imf_tvar='BX_GSE-BY_GSM-BZ_GSM_joined',
    Np_tvar='proton_density',
    Vp_tvar='flow_speed',
    model='ts04',
    pressure_tvar='Pressure',
    speed=True,
    newname='ts04_par'
)

ts04_par = psp.get_data(params_name, xarray=True)
print(ts04_par)

# trace 時刻へ補間
ts04_par_i = ts04_par.interp(time=trace_times, method='nearest')

print(ts04_par_i)
print("parmod columns = [Pdyn, Dst, ByIMF, BzIMF, W1, W2, W3, W4, W5, W6]")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import geopack
from geopack import trace_vectorized, smgsm_vectorized

def make_tsy_parmod(
    model,
    trace_times,
    ts04_par=None,
    t96_base_par=None,
    t01_g=None,
    t89_iopt=None,
):
    """
    Tsyganenko model 用 parmod を trace_times に合わせて返す。

    Returns
    -------
    par_values : ndarray
        shape = (len(trace_times), 10)
    exname : str
        geopack/trace_vectorized に渡す exname
    columns : list[str]
        parmod の列名
    """
    model = model.lower()
    trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))
    nt = len(trace_times)

    if model in ["t04", "ts04"]:
        if ts04_par is None:
            raise ValueError("T04/TS04 requires ts04_par.")
        par_values = _interp_parmod_to_trace_times(ts04_par, trace_times)
        exname = "t04"
        columns = ["Pdyn", "Dst", "ByIMF", "BzIMF",
                   "W1", "W2", "W3", "W4", "W5", "W6"]

    elif model == "t96":
        if t96_base_par is None:
            raise ValueError("T96 requires t96_base_par.")
        base = _interp_parmod_to_trace_times(t96_base_par, trace_times)

        par_values = np.zeros((nt, 10), dtype=float)
        par_values[:, 0:4] = base[:, 0:4]

        exname = "t96"
        columns = ["Pdyn", "Dst", "ByIMF", "BzIMF",
                   "unused5", "unused6", "unused7", "unused8", "unused9", "unused10"]

    elif model == "t01":
        if t96_base_par is None:
            raise ValueError("T01 requires t96_base_par.")
        base = _interp_parmod_to_trace_times(t96_base_par, trace_times)

        par_values = np.zeros((nt, 10), dtype=float)
        par_values[:, 0:4] = base[:, 0:4]

        if t01_g is not None:
            g = np.asarray(t01_g, dtype=float)
            if g.ndim == 1:
                if g.size != 2:
                    raise ValueError("1D t01_g must have length 2: [G1, G2].")
                g = np.tile(g, (nt, 1))
            if g.shape != (nt, 2):
                raise ValueError(f"t01_g must have shape ({nt}, 2), got {g.shape}.")
            par_values[:, 4:6] = g
        else:
            # 暫定。G1, G2 を厳密に使うなら別途計算する。
            par_values[:, 4:6] = 0.0

        exname = "t01"
        columns = ["Pdyn", "Dst", "ByIMF", "BzIMF",
                   "G1", "G2", "unused7", "unused8", "unused9", "unused10"]

    elif model == "t89":
        if t89_iopt is None:
            raise ValueError("T89 requires t89_iopt.")
        iopt = np.asarray(t89_iopt, dtype=float)

        if iopt.ndim == 0:
            iopt = np.full(nt, float(iopt))
        elif iopt.ndim == 1 and iopt.size == nt:
            pass
        else:
            raise ValueError(f"t89_iopt must be scalar or shape ({nt},), got {iopt.shape}.")

        par_values = np.zeros((nt, 10), dtype=float)
        par_values[:, 0] = iopt

        exname = "t89"
        columns = ["iopt", "unused2", "unused3", "unused4",
                   "unused5", "unused6", "unused7", "unused8", "unused9", "unused10"]

    else:
        raise ValueError("model must be one of 't89', 't96', 't01', 't04'/'ts04'.")

    return par_values, exname, columns

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import geopack
from geopack import trace_vectorized, smgsm_vectorized


def datetime64_to_unix_seconds(t):
    """
    pandas.Timestamp, numpy.datetime64, str などを Unix seconds に変換する。
    geopack.recalc に渡す用。
    """
    return pd.Timestamp(t).timestamp()


def unpack_path(xx, yy, zz):
    """
    trace_vectorized(return_full_path=True) の戻り値を 1 本分の path に整形する。
    masked array でも ndarray でも処理できるようにする。
    """
    xx = np.ma.asarray(xx).squeeze()
    yy = np.ma.asarray(yy).squeeze()
    zz = np.ma.asarray(zz).squeeze()

    mask = (
        np.ma.getmaskarray(xx)
        | np.ma.getmaskarray(yy)
        | np.ma.getmaskarray(zz)
        | ~np.isfinite(np.ma.filled(xx, np.nan))
        | ~np.isfinite(np.ma.filled(yy, np.nan))
        | ~np.isfinite(np.ma.filled(zz, np.nan))
    )

    x = np.ma.filled(xx, np.nan)[~mask]
    y = np.ma.filled(yy, np.nan)[~mask]
    z = np.ma.filled(zz, np.nan)[~mask]

    return x, y, z


def _get_time_values(da):
    """
    xarray.DataArray から time 座標を取り出す。
    """
    if "time" not in da.coords:
        raise ValueError("Input position DataArray must have a 'time' coordinate.")
    return pd.DatetimeIndex(pd.to_datetime(da["time"].values))


def _interp_position_to_trace_times(pos_gsm, trace_times):
    """
    pos_gsm を trace_times に補間する。
    pos_gsm: xarray.DataArray(time, 3), GSM, R_E
    """
    trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))

    if not isinstance(pos_gsm, xr.DataArray):
        raise TypeError("pos_gsm must be an xarray.DataArray.")

    if pos_gsm.ndim != 2 or pos_gsm.shape[1] != 3:
        raise ValueError("pos_gsm must have shape (time, 3).")

    pos_i = pos_gsm.interp(time=trace_times, method="nearest")

    return pos_i


def _interp_parmod_to_trace_times(ts04_par, trace_times):
    """
    ts04_par を trace_times に補間する。
    ts04_par: xarray.DataArray(time, 10) or ndarray(len(trace_times), 10)
    """
    trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))

    if isinstance(ts04_par, xr.DataArray):
        if "time" in ts04_par.coords:
            par_i = ts04_par.interp(time=trace_times, method="nearest")
            par_values = np.asarray(par_i.values, dtype=float)
        else:
            par_values = np.asarray(ts04_par.values, dtype=float)
    else:
        par_values = np.asarray(ts04_par, dtype=float)

    if par_values.ndim == 1:
        if par_values.size != 10:
            raise ValueError("1D parmod must have length 10.")
        par_values = np.tile(par_values, (len(trace_times), 1))

    if par_values.shape != (len(trace_times), 10):
        raise ValueError(
            f"parmod shape must be ({len(trace_times)}, 10), "
            f"but got {par_values.shape}."
        )

    return par_values


def trace_field_lines_gsm(
    pos_gsm,
    parmod,
    trace_times=None,
    sat_name=None,
    exname="t04",
    inname="igrf",
    rlim=30.0,
    r0=1.015,
    maxloop=5000,
    directions=(+1, -1),
    strict_scalar_models=False,
    verbose=True,
):
    """
    GSM 座標の衛星位置から field line を trace し、GSM path と SM path を返す。

    Parameters
    ----------
    pos_gsm : xarray.DataArray
        衛星位置。shape は (time, 3)。
        座標系は GSM、単位は R_E を想定。
    ts04_par : xarray.DataArray or ndarray
        T04/TS04 parmod。
        shape は (time, 10) または (10,)。
        成分順は [Pdyn, Dst, ByIMF, BzIMF, W1, W2, W3, W4, W5, W6]。
    trace_times : array-like or None
        trace する時刻。
        None の場合は pos_gsm.time をそのまま使う。
    sat_name : str or None
        出力 dict に入れる衛星名。
    exname : str
        外部磁場モデル。T04 の場合は "t04"。
    inname : str
        内部磁場モデル。"igrf" または "dip"。
    rlim : float
        外側境界 [R_E]。
    r0 : float
        内側境界 [R_E]。
    maxloop : int
        trace_vectorized の最大 step 数。
    directions : tuple
        trace 方向。通常は (+1, -1)。
    strict_scalar_models : bool
        geopack-vectorize の strict_scalar_models に渡す。
    verbose : bool
        True の場合、各時刻の status を print する。

    Returns
    -------
    field_lines : list[dict]
        各時刻の field line。
        各 dict は time, x_gsm, y_gsm, z_gsm, x_sm, y_sm, z_sm,
        x0_gsm, y0_gsm, z0_gsm, x0_sm, y0_sm, z0_sm, status, parmod を持つ。
    """

    if trace_times is None:
        trace_times = _get_time_values(pos_gsm)
    else:
        trace_times = pd.DatetimeIndex(pd.to_datetime(trace_times))

    pos_i = _interp_position_to_trace_times(pos_gsm, trace_times)
    par_values = _interp_parmod_to_trace_times(parmod, trace_times)

    field_lines = []

    for it, t in enumerate(trace_times):
        x0, y0, z0 = np.asarray(pos_i.values[it, :], dtype=float)
        parmod = np.asarray(par_values[it, :], dtype=float)

        if not np.all(np.isfinite([x0, y0, z0])) or not np.all(np.isfinite(parmod)):
            if verbose:
                print(f"skip {t}: NaN in position or parmod")
                print("position =", [x0, y0, z0])
                print("parmod   =", parmod)
            continue

        geopack.recalc(datetime64_to_unix_seconds(t))

        paths_gsm = []

        for direction in directions:
            try:
                ret = trace_vectorized(
                    x0, y0, z0,
                    dir=direction,
                    rlim=rlim,
                    r0=r0,
                    parmod=parmod,
                    exname=exname,
                    inname=inname,
                    maxloop=maxloop,
                    return_full_path=True,
                    strict_scalar_models=strict_scalar_models
                )

                # geopack-vectorize の版差対策:
                # こちらの環境では
                # xf, yf, zf, xx, yy, zz, status
                # ただし別実装では status の位置が違う可能性がある。
                if len(ret) >= 7:
                    xf, yf, zf, xx, yy, zz, status = ret[:7]
                else:
                    raise RuntimeError(
                        f"Unexpected trace_vectorized return length: {len(ret)}"
                    )

                x_path, y_path, z_path = unpack_path(xx, yy, zz)

                if len(x_path) > 1:
                    paths_gsm.append(
                        (
                            x_path,
                            y_path,
                            z_path,
                            int(np.asarray(status).squeeze())
                        )
                    )

            except Exception as e:
                if verbose:
                    print(f"trace failed {t}, dir={direction}: {e}")

        if len(paths_gsm) == 0:
            if verbose:
                print(f"failed {t}: no valid path")
            continue

        # 2方向を結合する。
        # directions=(+1, -1) を想定して、後半の path を反転してつなぐ。
        if len(paths_gsm) == 2:
            x1, y1, z1, s1 = paths_gsm[0]
            x2, y2, z2, s2 = paths_gsm[1]

            x_gsm = np.r_[x2[::-1], x1[1:]]
            y_gsm = np.r_[y2[::-1], y1[1:]]
            z_gsm = np.r_[z2[::-1], z1[1:]]
            status = (s1, s2)

        else:
            x_gsm, y_gsm, z_gsm, s1 = paths_gsm[0]
            status = (s1,)

        field_lines.append({
            "sat_name": sat_name,
            "time": t,
            "x_gsm": np.asarray(x_gsm),
            "y_gsm": np.asarray(y_gsm),
            "z_gsm": np.asarray(z_gsm),
            "x0_gsm": float(x0),
            "y0_gsm": float(y0),
            "z0_gsm": float(z0),
            "status": status,
            "parmod": parmod
        })

        if verbose:
            prefix = f"{sat_name}: " if sat_name is not None else ""
            print(prefix + f"{t} status {status} npts {len(x_gsm)}")

    return field_lines

In [ ]:
parmod_t04, exname_t04, cols_t04 = make_tsy_parmod(
    model="t04",
    trace_times=trace_times,
    ts04_par=ts04_par,
)

arase_fl_t04 = trace_field_lines_gsm(
    pos_gsm=arase_gsm_mid,
    parmod=parmod_t04,
    trace_times=trace_times,
    sat_name="Arase",
    exname=exname_t04,
)

tha_fl_t04 = trace_field_lines_gsm(
    pos_gsm=tha_gsm_mid,
    parmod=parmod_t04,
    trace_times=trace_times,
    sat_name="THEMIS-A",
    exname=exname_t04,
)

In [ ]:
parmod_t96, exname_t96, cols_t96 = make_tsy_parmod(
    model="t96",
    trace_times=trace_times,
    t96_base_par=ts04_par,
)

arase_fl_t96 = trace_field_lines_gsm(
    pos_gsm=arase_gsm_mid,
    parmod=parmod_t96,
    trace_times=trace_times,
    sat_name="Arase",
    exname=exname_t96,
)

tha_fl_t96 = trace_field_lines_gsm(
    pos_gsm=tha_gsm_mid,
    parmod=parmod_t96,
    trace_times=trace_times,
    sat_name="THEMIS-A",
    exname=exname_t96,
)

In [ ]:
# ============================================================
# Field lines for each spacecraft
# ============================================================

def plot_field_lines_on_3panels(
    field_lines,
    ax_xy,
    ax_xz,
    ax_yz,
    color,
    label=None,
    lw=1.2,
    alpha=0.75,
    ls='--',
    zorder=2,
    mark_footpoints=False,
):
    """
    field_lines の x_sm, y_sm, z_sm を
    (X,Y), (X,Z), (Y,Z) の3 panel に重ねる。
    label は最初の1本だけに付ける。
    """
    for i, fl in enumerate(field_lines):
        this_label = label if i == 0 else None

        ax_xy.plot(
            fl["x_gsm"], fl["y_gsm"],
            color=color, linestyle=ls, lw=lw, alpha=alpha,
            label=this_label, zorder=zorder
        )
        ax_xz.plot(
            fl["x_gsm"], fl["z_gsm"],
            color=color, linestyle=ls, lw=lw, alpha=alpha,
            zorder=zorder
        )
        ax_yz.plot(
            fl["y_gsm"], fl["z_gsm"],
            color=color, linestyle=ls, lw=lw, alpha=alpha,
            zorder=zorder
        )

        if mark_footpoints:
            # 両端を小さく表示。不要なら False のままでよい。
            ax_xy.scatter(
                [fl["x_gsm"][0], fl["x_gsm"][-1]],
                [fl["y_gsm"][0], fl["y_gsm"][-1]],
                color=color, s=12, alpha=alpha, zorder=zorder + 1
            )
            ax_xz.scatter(
                [fl["x_gsm"][0], fl["x_gsm"][-1]],
                [fl["z_gsm"][0], fl["z_gsm"][-1]],
                color=color, s=12, alpha=alpha, zorder=zorder + 1
            )
            ax_yz.scatter(
                [fl["y_gsm"][0], fl["y_gsm"][-1]],
                [fl["z_gsm"][0], fl["z_gsm"][-1]],
                color=color, s=12, alpha=alpha, zorder=zorder + 1
            )

In [ ]:
def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

fig = plt.figure(figsize=(11, 11), dpi=300)

ax1 = fig.add_subplot(221)
ax1.plot(themis_a_pos_gsm_x, themis_a_pos_gsm_y, color='magenta', label='THEMIS-A', linewidth=3)
ax1.scatter(themis_a_pos_gsm_x[0],  themis_a_pos_gsm_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax1.scatter(themis_a_pos_gsm_x[-1], themis_a_pos_gsm_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax1.plot(Arase_pos_gsm_x, Arase_pos_gsm_y, color='green', label='Arase', linewidth=3)
ax1.scatter(Arase_pos_gsm_x[0],  Arase_pos_gsm_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax1.scatter(Arase_pos_gsm_x[-1], Arase_pos_gsm_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒GSM '+ r'[$R_{\mathrm{E}}$]', 'Y‒GSM '+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -11)
ax1.set_ylim(11, -1)
ax1.set_title('(X, Y)')


ax2 = fig.add_subplot(222)
ax2.plot(themis_a_pos_gsm_x, themis_a_pos_gsm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax2.scatter(themis_a_pos_gsm_x[0],  themis_a_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax2.scatter(themis_a_pos_gsm_x[-1], themis_a_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax2.plot(Arase_pos_gsm_x, Arase_pos_gsm_z, color='green', label='Arase', linewidth=3)
ax2.scatter(Arase_pos_gsm_x[0],  Arase_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax2.scatter(Arase_pos_gsm_x[-1], Arase_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax2, 'X‒GSM '+ r'[$R_{\mathrm{E}}$]', 'Z‒GSM '+ r'[$R_{\mathrm{E}}$]')
ax2.set_xlim(1, -11)
ax2.set_ylim(-6, 6)
ax2.set_title('(X, Z)')

ax3 = fig.add_subplot(223)
ax3.plot(themis_a_pos_gsm_y, themis_a_pos_gsm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax3.scatter(themis_a_pos_gsm_y[0],  themis_a_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax3.scatter(themis_a_pos_gsm_y[-1], themis_a_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax3.plot(Arase_pos_gsm_y, Arase_pos_gsm_z, color='green', label='Arase', linewidth=3)
ax3.scatter(Arase_pos_gsm_y[0],  Arase_pos_gsm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax3.scatter(Arase_pos_gsm_y[-1], Arase_pos_gsm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax3, 'Y‒GSM '+ r'[$R_{\mathrm{E}}$]', 'Z‒GSM '+ r'[$R_{\mathrm{E}}$]')
ax3.set_xlim(10, -1)
ax3.set_ylim(-5.5, 5.5)
ax3.set_title('(Y, Z)')

for ax in (ax1, ax2, ax3):
    th = np.linspace(0, 2*np.pi, 256)
    ax.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
    if ax == ax1 or ax == ax2 or ax == ax4:
        # 円の中のx<0の部分を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
    if ax == ax3:
        # 円の中を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)>0), color='k')

    if ax == ax1:
        ax.text(-0.25, 1.1, '(a)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax2:
        ax.text(-0.25, 1.1, '(b)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax3:
        ax.text(-0.25, 1.1, '(c)', transform=ax.transAxes, verticalalignment='top')

plot_field_lines_on_3panels(
    tha_fl_t96,
    ax1, ax2, ax3,
    color='magenta',
    label='THEMIS-A field line',
    lw=1.1,
    alpha=0.65,
    ls='--',
    zorder=2
)

plot_field_lines_on_3panels(
    arase_fl_t96,
    ax1, ax2, ax3,
    color='green',
    label='Arase field line',
    lw=1.1,
    alpha=0.65,
    ls='--',
    zorder=2
)

fig.tight_layout()
plt.show()

fig.savefig(r"/mnt/j/KAW_observation/probe_trajectory_T96_gsm.pdf")
fig.savefig(r"/mnt/j/KAW_observation/probe_trajectory_T96_gsm.png")